# 04c — Robustness Check

Test whether LLM results generalize by running the **same model** on a **different 100-loan sample** from the test set.

**Goal:** Confirm findings from 04a aren't artifacts of the specific sample chosen.

## Setup

In [ ]:
import os
import pandas as pd
import numpy as np

from llm_utils import (
    load_llm_sample, sample_new_batch, run_ml_on_sample, run_llm_experiment,
    evaluate_predictions, compare_results, RESULTS_DIR
)

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
API_PROVIDER = "gemini"
MODEL_NAME   = "gemini-2.5-flash"  # Same model as 04a
LABEL        = "Gemini 2.5 Flash"
INCLUDE_DESC = True

## Sample New Batch

In [ ]:
# Load original sample for comparison
original_sample = load_llm_sample()

# Sample a new, non-overlapping batch of 100 loans
new_sample = sample_new_batch(n=100, random_state=99)

print(f"Original sample: {len(original_sample)} loans")
print(f"New sample:      {len(new_sample)} loans")
print(f"\nNew sample class distribution:")
print(new_sample['loan_status'].value_counts())
print(f"\nOriginal sample class distribution:")
print(original_sample['loan_status'].value_counts())

## Run XGBoost & LLM on New Sample

In [ ]:
# XGBoost on new sample
xgb_probs_new, xgb_preds_new = run_ml_on_sample(new_sample)
y_true_new = new_sample['loan_status'].values

print(f"XGBoost predictions ready: {len(xgb_preds_new)} samples")

In [ ]:
# LLM on new sample
new_result = run_llm_experiment(
    new_sample,
    api_provider=API_PROVIDER,
    model_name=MODEL_NAME,
    include_desc=INCLUDE_DESC,
    label=f"{LABEL} (new batch)",
)

## Compare: Original vs New Batch

In [ ]:
# Load original results from 04a
original_metrics_path = f"{RESULTS_DIR}/04a_model_comparison_metrics.csv"
if os.path.exists(original_metrics_path):
    orig_metrics = pd.read_csv(original_metrics_path, index_col=[0, 1])
    print("Original sample results (from 04a):")
    print(orig_metrics.to_string())
else:
    print("04a results not found — run 04a first for comparison")

# New sample results
xgb_metrics_new = evaluate_predictions(y_true_new, xgb_preds_new.tolist(), label="XGBoost (new batch)")

desc_tag = 'with_desc' if INCLUDE_DESC else 'no_desc'
print(f"\n{'='*60}")
print(f"Side-by-side: Original vs New Batch ({desc_tag})")
print(f"{'='*60}")

comparison_rows = [
    {'batch': 'New', 'model': 'XGBoost', **xgb_metrics_new},
    {'batch': 'New', 'model': LABEL, **new_result['metrics']},
]
new_df = pd.DataFrame(comparison_rows)
print(new_df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

# Visual comparison if 04a results exist
if os.path.exists(original_metrics_path):
    orig = orig_metrics.reset_index()
    # Find matching rows
    desc_cond = 'with_desc' if INCLUDE_DESC else 'no_desc'
    orig_llm = orig[(orig['model'] == LABEL) & (orig['condition'] == desc_cond)]
    orig_xgb = orig[orig['model'] == 'XGBoost']

    labels = ['XGBoost\n(original)', 'XGBoost\n(new)', f'{LABEL}\n(original)', f'{LABEL}\n(new)']
    accuracies = [
        orig_xgb['accuracy'].values[0] if len(orig_xgb) else 0,
        xgb_metrics_new['accuracy'],
        orig_llm['accuracy'].values[0] if len(orig_llm) else 0,
        new_result['metrics']['accuracy'],
    ]

    colors = ['#2196F3', '#64B5F6', '#FF9800', '#FFB74D']
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(labels, [a * 100 for a in accuracies], color=colors)
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Robustness Check: Original vs New Sample')
    ax.set_ylim(0, 100)
    for bar, acc in zip(bars, accuracies):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{acc*100:.1f}%', ha='center')
    plt.tight_layout()
    plt.show()

## Export Results

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

# Save new batch predictions
comparison = compare_results(y_true_new, new_result['predictions'],
                             xgb_preds_new.tolist(), new_result['reasonings'])
comparison.to_csv(f"{RESULTS_DIR}/04c_robustness_new_batch_results.csv", index=False)

# Save metrics comparison
robustness_summary = pd.DataFrame([
    {'batch': 'New', 'model': 'XGBoost', **xgb_metrics_new},
    {'batch': 'New', 'model': LABEL, **new_result['metrics']},
])
robustness_summary.to_csv(f"{RESULTS_DIR}/04c_robustness_metrics.csv", index=False)

print(f"Results saved to {RESULTS_DIR}/")